In [6]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from uuid import uuid4

In [4]:

from langchain_core.documents import Document
# 12 documents: ML (highly relevant to query), economics (somewhat relevant), cooking and sports (off-topic)
# The threshold filter will progressively cut out the lower-scoring documents
docs = [
    Document(page_content="Supervised learning trains models on labeled input-output pairs to predict unseen data.", metadata={"topic": "ml"}),
    Document(page_content="Neural networks learn by adjusting weights through backpropagation and gradient descent.", metadata={"topic": "ml"}),
    Document(page_content="Overfitting occurs when a model memorises training data and performs poorly on new data.", metadata={"topic": "ml"}),
    Document(page_content="Training data quality and quantity are the most important factors in model performance.", metadata={"topic": "ml"}),
    Document(page_content="Cross-validation splits data into folds to evaluate model generalisation more reliably.", metadata={"topic": "ml"}),
    Document(page_content="Inflation is the rate at which the general level of prices for goods and services rises over time.", metadata={"topic": "economics"}),
    Document(page_content="GDP measures the total monetary value of all goods and services produced within a country.", metadata={"topic": "economics"}),
    Document(page_content="Interest rates set by central banks influence borrowing costs and consumer spending.", metadata={"topic": "economics"}),
    Document(page_content="Caramelisation occurs when sugar is heated above 160°C, creating complex flavour compounds.", metadata={"topic": "cooking"}),
    Document(page_content="Fermentation uses microorganisms to convert sugars into alcohol or acids, preserving food.", metadata={"topic": "cooking"}),
    Document(page_content="A marathon is a long-distance race of exactly 42.195 kilometres, run on roads.", metadata={"topic": "sports"}),
    Document(page_content="Tennis scoring follows a love, 15, 30, 40, game sequence, with deuce at 40-40.", metadata={"topic": "sports"}),
]



In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
vectorstore = Chroma(
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_configuration={
        "hnsw": {
            "space": "cosine"      # cosine, L2, ip
        }
    }
)

In [19]:
# add documents to vector store

ids_added = vectorstore.add_documents(
    documents=docs,
    ids=[str(uuid4()) for _ in range(len(docs))]
)

In [35]:
query = "How does ML model training work?"

threshold = 0.2

retriever = vectorstore.as_retriever(
    search_type = 'similarity_score_threshold',
    search_kwargs={'score_threshold': threshold}
)

result = retriever.invoke(query)
result

No relevant docs were retrieved using the relevance score threshold 0.2


[]

In [26]:
# see similarity scores for each document in the result

query = "How does ML model training work?"

result_with_score = vectorstore.similarity_search_with_score(query)
result_with_score

[(Document(id='a15b4f43-bbd4-41bb-8f12-71e561459ff8', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'),
  1.1362870931625366),
 (Document(id='e5c0b12f-57e9-4584-9e53-e1cb026d7793', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.'),
  1.1442087888717651),
 (Document(id='efc412a2-da05-4543-8cbe-aa8007768439', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.'),
  1.1442742347717285),
 (Document(id='e2e238e6-64f6-4407-871e-f5496609265d', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'),
  1.2154669761657715)]

In [38]:
query = "How does ML model training work?"

result_with_score = vectorstore.similarity_search_with_relevance_scores(query)
print('result_with_relevance_scores')
result_with_score

result_with_relevance_scores


[(Document(id='a15b4f43-bbd4-41bb-8f12-71e561459ff8', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'),
  0.1965236910500201),
 (Document(id='e5c0b12f-57e9-4584-9e53-e1cb026d7793', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.'),
  0.19092220629552825),
 (Document(id='efc412a2-da05-4543-8cbe-aa8007768439', metadata={'topic': 'ml'}, page_content='Supervised learning trains models on labeled input-output pairs to predict unseen data.'),
  0.19087592905586326),
 (Document(id='e2e238e6-64f6-4407-871e-f5496609265d', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'),
  0.14053505884487527)]

In [ ]:
query = "Training data quality"

print(vectorstore.similarity_search_with_score(query))
print(vectorstore.similarity_search_with_relevance_scores(query))

[(Document(id='a15b4f43-bbd4-41bb-8f12-71e561459ff8', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performance.'), 0.5683385133743286), (Document(id='0475a40d-222d-4645-a42f-6db04f2d390c', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'), 1.1344406604766846), (Document(id='e2e238e6-64f6-4407-871e-f5496609265d', metadata={'topic': 'ml'}, page_content='Overfitting occurs when a model memorises training data and performs poorly on new data.'), 1.1344566345214844), (Document(id='0c3c7f51-01b3-41d9-baff-bf18bbfd4003', metadata={'topic': 'ml'}, page_content='Cross-validation splits data into folds to evaluate model generalisation more reliably.'), 1.2952724695205688)]
[(Document(id='a15b4f43-bbd4-41bb-8f12-71e561459ff8', metadata={'topic': 'ml'}, page_content='Training data quality and quantity are the most important factors in model performa

: 